In [0]:

# Load a file into a dataframe
#df = spark.read.load('/data/mydata.csv', format='csv', header=True)
df = spark.read.load('dbfs:/FileStore/test/RSample/ProductsDelta.csv', format='csv', header=True)

# Save the dataframe as a delta table
#delta_table_path = "/delta/mydata"
delta_table_path = "dbfs:/FileStore/test/RSample/ProductsDelta.csv"

df.write.format("delta").save(delta_table_path)

#You can replace an existing Delta Lake table with the contents of a dataframe by using the overwrite mode, as shown here:
new_df.write.format("delta").mode("overwrite").save(delta_table_path)

#You can also add rows from a dataframe to an existing table by using the append mode:
new_rows_df.write.format("delta").mode("append").save(delta_table_path)



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-560069847879941>:9
      5 # Save the dataframe as a delta table
      6 #delta_table_path = "/delta/mydata"
      7 delta_table_path = "dbfs:/FileStore/test/RSample/ProductsDelta.csv"
----> 9 df.write.format("delta").save(delta_table_path)

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1397, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
   1395     self._jwrite.save()
   1396 else:
-> 1397     self._jwrite.save(path)



In [0]:
#Making conditional updates

from delta.tables import *
from pyspark.sql.functions import *

# Create a deltaTable object
deltaTable = DeltaTable.forPath(spark, delta_table_path)

# Update the table (reduce price of accessories by 10%)
deltaTable.update(
    condition = "Category == 'Accessories'",
    set = { "Price": "Price * 0.9" })


In [0]:
#Querying a previous version of a table
df = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)


In [0]:
#Alternatively, you can specify a timestamp by using the timestampAsOf option:
df = spark.read.format("delta").option("timestampAsOf", '2022-01-01').load(delta_table_path)


In [0]:

#Creating catalog tables
##Creating a catalog table from a dataframe

# Save a dataframe as a managed table
df.write.format("delta").saveAsTable("MyManagedTable")

## specify a path option to save as an external table
df.write.format("delta").option("path", "/mydata").saveAsTable("MyExternalTable")


In [0]:
#Creating a catalog table using SQL
##
spark.sql("CREATE TABLE MyExternalTable USING DELTA LOCATION '/mydata'")

%sql

CREATE TABLE MyExternalTable
USING DELTA
LOCATION '/mydata'


In [0]:
#Defining the table schema

%sql

CREATE TABLE ManagedSalesOrders
(
    Orderid INT NOT NULL,
    OrderDate TIMESTAMP NOT NULL,
    CustomerName STRING,
    SalesTotal FLOAT NOT NULL
)
USING DELTA

In [0]:
%sql
--from delta.tables import *

use dw_xle



In [0]:
#Using the DeltaTableBuilder API

from delta.tables import *

DeltaTable.create(spark) \
  .tableName("ManagedProducts") \
  #.tableName("default.ManagedProducts") \
  .addColumn("Productid", "INT") \
  .addColumn("ProductName", "STRING") \
  .addColumn("Category", "STRING") \
  .addColumn("Price", "FLOAT") \
  .execute()


  File <command-2638759489980523>, line 8
    .addColumn("Productid", "INT") \
    ^
IndentationError: unexpected indent


In [0]:
#Using catalog tables
%sql

SELECT orderid, salestotal
FROM ManagedSalesOrders


  File <command-560069847879954>:4
    SELECT orderid, salestotal
           ^
SyntaxError: invalid syntax


In [0]:
funcs = []

for i in range(4):
  print('i value ', i)
  funcs.append(lambda: i)
  # print(funcs[i])

for f in funcs:
  print(f())


i value  0
i value  1
i value  2
i value  3
3
3
3
3
